In [22]:
from pathlib import Path
import os
import importlib
import awkward as ak
import numpy as np

from mltau.tools.evaluation import kinematics as k
from mltau.tools.evaluation import tagging as t
from mltau.tools.evaluation import charge_id as c
from mltau.tools.evaluation import decay_mode as d

k = importlib.reload(k)
t = importlib.reload(t)
c = importlib.reload(c)
d = importlib.reload(d)



from mltau.tools.general import reinitialize_p4
from mltau.tools.evaluation import inference
from mltau.models import SingleParTau_module


In [23]:
from hydra import compose, initialize

with initialize(version_base=None, config_path="../config", job_name="test_app"):
    cfg = compose(config_name="main")

cfg.dataset.data_dir = "/home/laurits/0509_dsinphi_to_sindphi/"
print("cfg.dataset.data_dir:", cfg.dataset.data_dir)


cfg.dataset.data_dir: /home/laurits/0509_dsinphi_to_sindphi/


In [24]:
OUTPUT_DIR = "/home/norman/colors"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

CHARGE_FAKE_RATE_TARGET_EFFICIENCIES = [0.99, 0.95, 0.70]

SIGNAL_SAMPLE = "z"
BKG_SAMPLE = "qq"


In [25]:
def set_scaler(cfg, base_dir, task):
    cfg.training.input_scaling.scaler_path = os.path.join(
        base_dir,
        task,
        "scaler",
        "cand_feature_scaler.npz"
    )
    print(f"[DEBUG] Using scaler for {task}:")
    print(cfg.training.input_scaling.scaler_path)


# SingleParTau

In [26]:
SINGLE_PARTAU_TRAININGS_DIR = "/home/laurits/0509_SingleParTau/"


In [27]:
# Tagging
sTag_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "is_tau", "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
sTag_bkgData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "is_tau", "predictions", f"{BKG_SAMPLE}_test.parquet")
)
set_scaler(cfg, SINGLE_PARTAU_TRAININGS_DIR, "is_tau")
sTag_evaluator = t.TaggerEvaluator(
    signal_predictions=sTag_sigData.tau_tagging_score,
    signal_gen_tau_p4=sTag_sigData.gen_jet_tau_p4,
    signal_reco_jet_p4=sTag_sigData.reco_jet_p4,
    bkg_predictions=sTag_bkgData.tau_tagging_score,
    bkg_gen_jet_p4=sTag_bkgData.gen_jet_p4,
    bkg_reco_jet_p4=sTag_bkgData.reco_jet_p4,
    cfg=cfg,
    sample=SIGNAL_SAMPLE,
    algorithm="SingleParTau",
)


[DEBUG] Using scaler for is_tau:
/home/laurits/0509_SingleParTau/is_tau/scaler/cand_feature_scaler.npz


In [28]:
SINGLE_PARTAU_TRAININGS_DIR = "/home/laurits/0509_SingleParTau/"


In [29]:
# Decay mode
sDM_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "decay_mode", "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
set_scaler(cfg, SINGLE_PARTAU_TRAININGS_DIR, "decay_mode")
sDM_evaluator = d.DecayModeEvaluator(
    pred_proba=sDM_sigData.tau_decay_mode_probs,
    truth=sDM_sigData.gen_jet_tau_decaymode,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="SingleParTau"
)


[DEBUG] Using scaler for decay_mode:
/home/laurits/0509_SingleParTau/decay_mode/scaler/cand_feature_scaler.npz


In [30]:
# Charge
sCh_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "charge", "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
# set_scaler(cfg, SINGLE_PARTAU_TRAININGS_DIR, "charge")
sCh_evaluator = c.ChargeIdEvaluator(
    predicted=sCh_sigData.tau_charge_score,
    truth=sCh_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=sCh_sigData.gen_jet_tau_p4,
    reco_jet_p4s=sCh_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="SingleParTau",
)


In [31]:
# Kinematics
sKin_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "kinematics", "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
# set_scaler(cfg, SINGLE_PARTAU_TRAININGS_DIR, "kin")
sKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=sKin_sigData.tau_p4,
    true_p4=sKin_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="SingleParTau",
    sample_name=SIGNAL_SAMPLE
)


# MultiParTau

In [32]:
# MULTI_PARTAU_TRAININGS_DIR = "/home/laurits/0509_sindphi_training/"
MULTI_PARTAU_TRAININGS_DIR = "/home/laurits/0528_adjust_lr/"


In [33]:
multiParTau_sigData = ak.from_parquet(
    os.path.join(MULTI_PARTAU_TRAININGS_DIR, "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
multiParTau_bkgData = ak.from_parquet(
    os.path.join(MULTI_PARTAU_TRAININGS_DIR, "predictions", f"{BKG_SAMPLE}_test.parquet")
)

mTag_evaluator = t.TaggerEvaluator(
    signal_predictions=multiParTau_sigData.tau_tagging_score,
    signal_gen_tau_p4=multiParTau_sigData.gen_jet_tau_p4,
    signal_reco_jet_p4=multiParTau_sigData.reco_jet_p4,
    bkg_predictions=multiParTau_bkgData.tau_tagging_score,
    bkg_gen_jet_p4=multiParTau_bkgData.gen_jet_p4,
    bkg_reco_jet_p4=multiParTau_bkgData.reco_jet_p4,
    cfg=cfg,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau",
)

mDM_evaluator = d.DecayModeEvaluator(
    pred_proba=multiParTau_sigData.tau_decay_mode_probs,
    truth=multiParTau_sigData.gen_jet_tau_decaymode,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau"
)

mCh_evaluator = c.ChargeIdEvaluator(
    predicted=multiParTau_sigData.tau_charge_score,
    truth=multiParTau_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=multiParTau_sigData.gen_jet_tau_p4,
    reco_jet_p4s=multiParTau_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau",
)

mKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=multiParTau_sigData.tau_p4,
    true_p4=multiParTau_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="MultiParTau",
    sample_name=SIGNAL_SAMPLE
)


# HPS

In [34]:
HPS_PRED_DIR = "/home/laurits/HPS"


In [35]:
hps_sigData = ak.from_parquet(
    os.path.join(HPS_PRED_DIR, "hps_z.parquet")
)
hps_bkgData = ak.from_parquet(
    os.path.join(HPS_PRED_DIR, "QQ", "_tmp_merge", "chunk_0001.parquet" )
)


hCh_evaluator = c.HardLabelChargeIdEvaluator(
    predicted=hps_sigData.tau_charge,
    truth=hps_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=hps_sigData.gen_jet_tau_p4,
    reco_jet_p4s=hps_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="HPS",
)

hDM_evaluator = d.HardLabelDecayModeEvaluator(
    predicted=hps_sigData.tau_decaymode,
    truth=hps_sigData.gen_jet_tau_decaymode,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="HPS"
)

hKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=hps_sigData.tau_p4s,
    true_p4=hps_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="HPS",
    sample_name=SIGNAL_SAMPLE
)


# RecoJet

In [36]:
def calculate_qkappa(cand_p4, jet_p4, cand_charges, best_kappa=0.5):
    cand_pts = reinitialize_p4(cand_p4).pt
    jet_pts = reinitialize_p4(jet_p4).pt
    numerator = np.sum(cand_charges * cand_pts ** best_kappa, axis=1)
    denominator = jet_pts ** best_kappa
    qkappa_charge = numerator / np.where(denominator > 0, denominator, 1.0)
    qkappa_charge_score = np.clip(0.5 * (qkappa_charge + 1.0), 0.0, 1.0)
    return qkappa_charge_score

qkappa_input_path = os.path.join(HPS_PRED_DIR, f"{SIGNAL_SAMPLE}_test.parquet")
if os.path.exists(qkappa_input_path):
    qkappa_data = ak.from_parquet(qkappa_input_path)
else:
    qkappa_data = hps_sigData

cand_p4 = reinitialize_p4(qkappa_data.reco_cand_p4s)
jet_p4 = reinitialize_p4(qkappa_data.reco_jet_p4)
cand_charges = qkappa_data.reco_cand_charges

qkappa_charge_score = calculate_qkappa(cand_p4, jet_p4, cand_charges)

qkCh_evaluator = c.ChargeIdEvaluator(
    predicted=qkappa_charge_score,
    truth=qkappa_data.gen_jet_tau_charge,
    gen_jet_tau_p4s=qkappa_data.gen_jet_tau_p4,
    reco_jet_p4s=qkappa_data.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="QKappa",
)

rKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=sKin_sigData.reco_jet_p4,
    true_p4=sKin_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="RecoJet",
    sample_name=SIGNAL_SAMPLE
)


# Combined results

In [37]:
# Kinematics
kRESULTS_DIR = os.path.join(RESULTS_DIR, "kinematics")
os.makedirs(kRESULTS_DIR, exist_ok=True)
kme = k.KinematicsMultiEvaluator(kRESULTS_DIR, cfg, sample=SIGNAL_SAMPLE)
kme.combine_results([sKin_evaluator, rKin_evaluator, mKin_evaluator]) # hKin_evaluator
kme.save()

# # Charge
cRESULTS_DIR = os.path.join(RESULTS_DIR, "charge_id")
os.makedirs(cRESULTS_DIR, exist_ok=True)
cme = c.ChargeMultiEvaluator(
    cRESULTS_DIR,
    cfg,
    target_efficiencies=CHARGE_FAKE_RATE_TARGET_EFFICIENCIES,
)
cme.combine_results([sCh_evaluator, mCh_evaluator, qkCh_evaluator]) # hCh_evaluator
cme.save()

# # Tagging
tRESULTS_DIR = os.path.join(RESULTS_DIR, "tau_id")
os.makedirs(tRESULTS_DIR, exist_ok=True)
tme = t.TaggerMultiEvaluator(tRESULTS_DIR, cfg)
tme.combine_results([sTag_evaluator, mTag_evaluator])
tme.save()

# # Decay mode
dRESULTS_DIR = os.path.join(RESULTS_DIR, "decay_mode")
os.makedirs(dRESULTS_DIR, exist_ok=True)
dme = d.DecayModeMultiEvaluator(dRESULTS_DIR, cfg, sample=SIGNAL_SAMPLE)
dme.combine_results([sDM_evaluator, mDM_evaluator]) # hDM_evaluator
dme.save()


/home/norman/model/ml-tau-model/mltau/tools/evaluation/kinematics.py:266: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, rows = plt.subplots(nrows=3, ncols=4, sharex="col", figsize=(16, 9))


# Losses (maybe we want to plot some losses?)

In [43]:
from tensorboard.backend.event_processing import event_accumulator

# log_dir = "/home/laurits/tmp/speedup_test2/tensorboard/ParTau_experiment/version_0/"

# ea = event_accumulator.EventAccumulator(log_dir)
# ea.Reload()

# # List available scalar tags
# print(ea.Tags()["scalars"])

# # Extract a specific scalar
# scalars = ea.Scalars("train_losses/decay_mode_loss")

# for s in scalars:
#     print(s.step, s.value)